# LeetCode 790 — Domino and Tromino Tiling bằng Python

Link đề: https://leetcode.com/problems/domino-and-tromino-tiling/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def numTilings(self, n: int) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên lấy modulo `10^9 + 7`). Notebook dùng phép **liệt kê vét cạn các cách lát** (trên bảng nhỏ) làm oracle độc lập, rồi đối chiếu các cách quy hoạch động, công thức truy hồi và nhân ma trận bằng `assert`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số cách lát kín bảng `2 × n` bằng hai loại gạch — **domino** `2 × 1` và **tromino hình chữ L** (3 ô) — được phép xoay; kết quả lấy modulo `10^9 + 7`.
2. Nắm các công cụ Python: danh sách trạng thái, `%` modulo và số nguyên lớn, tập hợp `frozenset` để chuẩn hoá một cách lát, đệ quy có nhớ `lru_cache`, nhân ma trận `3 × 3` và luỹ thừa nhanh.
3. Đi từ **liệt kê mọi cách lát bằng quay lui** → **quy hoạch động theo trạng thái cột** (4 trạng thái: đầy, thừa ô trên, thừa ô dưới, rỗng) → **công thức truy hồi `f(n) = 2·f(n−1) + f(n−3)`** → **nhân ma trận** (`O(log n)`).
4. Hiểu mấu chốt: khi lát từ trái sang phải, chỉ cần nhớ **cột hiện tại đã bị phủ những ô nào** (vì gạch dài nhất chỉ chiếm hai cột liền nhau); từ đó ra công thức truy hồi và chứng minh được.
5. Tránh các lỗi kinh điển: chỉ dùng domino (ra dãy Fibonacci), quên **hai hướng** của tromino (nhân đôi), sai điều kiện đầu (`f(0) = 1`), quên modulo, gộp nhầm hai trạng thái "chỉ phủ ô trên" và "chỉ phủ ô dưới", và đệ quy không nhớ gây chậm.

## 2. Đề bài & ví dụ

Có hai loại gạch: **domino** `2 × 1` và **tromino** hình chữ L (3 ô); cả hai có thể xoay. Cho số nguyên `n`, trả về **số cách lát kín bảng `2 × n`** (mọi ô đều được phủ, gạch không chồng nhau và không ra ngoài bảng), lấy modulo `10^9 + 7`.

Hai cách lát được coi là **khác nhau** khi và chỉ khi tồn tại hai ô kề nhau (4 hướng) mà trong đúng một cách lát, hai ô đó thuộc **cùng một viên gạch**. Nói đơn giản: hai cách lát là khác nhau nếu chúng chia bảng thành các viên gạch khác nhau.

**Ví dụ 1:** `n = 3` → `5`.

**Ví dụ 2:** `n = 1` → `1` (chỉ một domino đặt đứng).

**Ràng buộc:** `1 <= n <= 1000`.

Với `n = 3` có đúng 5 cách: (1) ba domino đứng; (2) một domino đứng rồi hai domino nằm; (3) hai domino nằm rồi một domino đứng; (4) và (5) hai tromino L ghép thành khối `2 × 3`, có hai cách ghép đối xứng nhau. Ta sẽ **liệt kê và vẽ** các cách này bằng code ở mục 4.

In [1]:
import time
import random
from functools import lru_cache
from itertools import product

MOD = 10 ** 9 + 7
ex1, ex2 = 3, 1
print("MOD =", MOD, "| ví dụ: n=3 ->", 5, ", n=1 ->", 1)

MOD = 1000000007 | ví dụ: n=3 -> 5 , n=1 -> 1


## 3. Kiến thức nền

### 3.1 Modulo và số nguyên lớn

Đáp án tăng rất nhanh (xấp xỉ `2,4^n`), với `n = 1000` có hàng trăm chữ số, nên đề yêu cầu lấy `% (10^9 + 7)`. Phép `%` tương thích với `+` và `×`: `(a + b) % M == ((a % M) + (b % M)) % M`. Vì vậy ta có thể lấy modulo **sau mỗi phép tính** để số luôn nhỏ. Python có số nguyên cỡ tuỳ ý nên ta còn có thể tính **đáp án chính xác** rồi mới `% MOD` để kiểm chứng.

In [2]:
a, b = 10 ** 9 + 6, 10 ** 9 + 5
print((a + b) % MOD, ((a % MOD) + (b % MOD)) % MOD)
print((a * b) % MOD, ((a % MOD) * (b % MOD)) % MOD)
print("2^200 có", len(str(2 ** 200)), "chữ số; 2^200 % MOD =", 2 ** 200 % MOD, "=", pow(2, 200, MOD))

1000000004 1000000004
2 2
2^200 có 61 chữ số; 2^200 % MOD = 499445072 = 499445072


### 3.2 Biểu diễn gạch và bảng bằng ô `(hàng, cột)`

Bảng `2 × n` có các ô `(r, c)` với `r ∈ {0, 1}` (hàng trên/dưới) và `c ∈ {0, …, n−1}`. Mỗi viên gạch là một **tập ô**:

- domino đứng: `{(0,c), (1,c)}`; domino nằm: `{(r,c), (r,c+1)}`;
- tromino L: lấy một hình vuông `2 × 2` (cột `c` và `c+1`) và **bỏ một trong 4 ô** → 4 hướng.

Một cách lát là một **tập các viên gạch** (`frozenset` của `frozenset` ô) — hai cách lát bằng nhau khi và chỉ khi chúng chia bảng thành cùng các viên gạch, đúng với định nghĩa "khác nhau" của đề.

In [3]:
def shapes_at(c):
    """Mọi viên gạch có thể đặt với cột trái nhất là cột c (không kiểm tra biên phải)."""
    out = []
    out.append(frozenset({(0, c), (1, c)}))                 # domino đứng
    for r in (0, 1):
        out.append(frozenset({(r, c), (r, c + 1)}))         # domino nằm ở hàng r
    square = [(0, c), (0, c + 1), (1, c), (1, c + 1)]
    for missing in square:                                  # tromino L: bỏ một ô của hình vuông 2x2
        out.append(frozenset(x for x in square if x != missing))
    return out


for s in shapes_at(0):
    print(sorted(s))
print("tổng số hướng đặt tại một cột:", len(shapes_at(0)), "(1 domino đứng + 2 domino nằm + 4 tromino)")

[(0, 0), (1, 0)]
[(0, 0), (0, 1)]
[(1, 0), (1, 1)]
[(0, 1), (1, 0), (1, 1)]
[(0, 0), (1, 0), (1, 1)]
[(0, 0), (0, 1), (1, 1)]
[(0, 0), (0, 1), (1, 0)]
tổng số hướng đặt tại một cột: 7 (1 domino đứng + 2 domino nằm + 4 tromino)


### 3.3 Đệ quy có nhớ và trạng thái

Nếu hàm đệ quy lặp lại cùng một **trạng thái** nhiều lần ta nhớ lại kết quả (`functools.lru_cache`). Ở bài này trạng thái là "đã lát xong các cột bên trái, và cột hiện tại bị phủ sẵn những ô nào" — chỉ có `4` khả năng cho mỗi cột nên số trạng thái là `4n`, rất nhỏ so với số cách lát (`~2,4^n`). Đó là bản chất của quy hoạch động: **đếm** các cách mà không **liệt kê** chúng.

### 3.4 Nhân ma trận và luỹ thừa nhanh

Nếu vector trạng thái ở bước `k + 1` bằng `M · (vector ở bước k)`, thì ở bước `n` bằng `Mⁿ · v₀`. `Mⁿ` tính trong `O(log n)` phép nhân ma trận bằng **bình phương liên tiếp** (`M^8 = ((M²)²)²`). Với ma trận cỡ `3 × 3`, mỗi phép nhân chỉ 27 phép nhân số.

In [4]:
def mat_mul(A, B, mod=None):
    n, m, p = len(A), len(B), len(B[0])
    C = [[sum(A[i][k] * B[k][j] for k in range(m)) for j in range(p)] for i in range(n)]
    return [[x % mod for x in row] for row in C] if mod else C


def mat_pow(M, e, mod=None):
    R = [[int(i == j) for j in range(len(M))] for i in range(len(M))]
    while e:
        if e & 1:
            R = mat_mul(R, M, mod)
        M = mat_mul(M, M, mod)
        e >>= 1
    return R


F = [[1, 1], [1, 0]]                       # ma trận Fibonacci
print([mat_pow(F, n)[0][1] for n in range(1, 11)], "<- Fibonacci bằng luỹ thừa ma trận")

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55] <- Fibonacci bằng luỹ thừa ma trận


## 4. Ý tưởng nền tảng / trực giác

Hãy **vẽ** các cách lát của `n = 3`. Ta kỳ vọng `5`. Ký hiệu mỗi viên gạch bằng một chữ cái; các ô cùng chữ thuộc cùng một viên.

In [5]:
def all_tilings(n):
    """Liệt kê mọi cách lát bảng 2 x n: trả về list các frozenset (mỗi phần tử là một viên gạch = frozenset ô)."""
    results = []

    def fill(covered, placed):
        # tìm ô trống đầu tiên theo thứ tự cột rồi hàng
        for c in range(n):
            for r in (0, 1):
                if (r, c) not in covered:
                    first = (r, c)
                    break
            else:
                continue
            break
        else:
            results.append(frozenset(placed))                # không còn ô trống: xong
            return
        c0 = first[1]
        for cc in (c0, c0 - 1) if c0 > 0 else (c0,):         # viên gạch có thể bắt đầu tại cột c0-1 (L che ô first)
            for shape in shapes_at(cc):
                if first in shape and all(0 <= c < n for _, c in shape) and not (shape & covered):
                    fill(covered | shape, placed + [shape])

    fill(frozenset(), [])
    return results


def draw(tiling, n):
    grid = [["." for _ in range(n)] for _ in range(2)]
    for idx, tile in enumerate(sorted(tiling, key=sorted)):
        for r, c in tile:
            grid[r][c] = chr(ord("A") + idx)
    return ["".join(row) for row in grid]


tilings3 = all_tilings(3)
print("số cách lát n = 3:", len(tilings3))
for t in tilings3:
    top, bottom = draw(t, 3)
    print(top, "/", bottom)

số cách lát n = 3: 5
ABC / ABC
ABB / ACC
AAB / CCB
ABB / AAB
AAB / ABB


Mỗi dòng là một cách lát (hàng trên / hàng dưới). Ta thấy: ba domino đứng `ABC/ABC`; một domino đứng cộng hai domino nằm (`ABB/ACC` và `AAB/CCB`, hai thứ tự); và hai tromino L ghép thành khối `2 × 3` (`ABB/AAB` và `AAB/ABB`, đối xứng nhau). Tổng `5`.

Ta cũng đếm cho `n = 1..10` để thấy quy luật tăng trưởng:

In [6]:
counts = [len(all_tilings(n)) for n in range(0, 11)]
print("n    :", list(range(0, 11)))
print("f(n) :", counts)
print("tỉ số f(n)/f(n-1):", [round(counts[i] / counts[i - 1], 3) for i in range(2, 11)])

n    : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
f(n) : [1, 1, 2, 5, 11, 24, 53, 117, 258, 569, 1255]
tỉ số f(n)/f(n-1): [2.0, 2.5, 2.2, 2.182, 2.208, 2.208, 2.205, 2.205, 2.206]


Dãy `1, 1, 2, 5, 11, 24, 53, 117, 258, 569, 1255` (với `f(0) = 1`: bảng rỗng có đúng 1 cách lát — "không làm gì"). Tỉ số tiến dần tới `≈ 2,2` (nghiệm của `x³ = 2x² + 1`). Dưới đây ta sẽ giải thích dãy này.

**Trực giác lát từ trái sang phải.** Nhìn cột trái nhất. Có ba loại gạch có thể phủ ô trên-trái:

1. Domino **đứng** — phủ cả cột; phần còn lại là bảng `2 × (n−1)`: `f(n−1)` cách.
2. Hai domino **nằm** chồng lên nhau — phủ hai cột; còn `2 × (n−2)`: `f(n−2)` cách.
3. Tromino **L** — phủ cả cột trái và **một ô** của cột kế; ô còn lại của cột kế phải được phủ bởi một viên **nằm kéo dài sang phải**, và cứ thế tạo thành một "chuỗi" các domino nằm so le trước khi khép lại bằng một L khác. Chuỗi này chính là thứ làm công thức phức tạp hơn domino thuần — ta sẽ gọn hoá bằng trạng thái cột.

## 5. Cách 1 — Brute Force: liệt kê mọi cách lát bằng quay lui

**Ý tưởng:** đúng như hàm `all_tilings` ở trên. Luôn tìm **ô trống đầu tiên** (theo thứ tự cột, rồi hàng), thử mọi viên gạch phủ ô đó mà không chồng lên gạch đã đặt và không ra ngoài bảng; đệ quy. Khi không còn ô trống, ghi nhận một cách lát.

Vì ta luôn lấp ô trống **đầu tiên**, mỗi cách lát được sinh ra **đúng một lần** (không có hoán vị trùng). Dùng bản đếm gọn, không lưu cách lát:

In [7]:
def count_brute(n):
    count = 0

    def fill(covered):
        nonlocal count
        first = None
        for c in range(n):
            for r in (0, 1):
                if (r, c) not in covered:
                    first = (r, c)
                    break
            if first:
                break
        if first is None:
            count += 1
            return
        c0 = first[1]
        for cc in ((c0, c0 - 1) if c0 > 0 else (c0,)):
            for shape in shapes_at(cc):
                if first in shape and all(0 <= c < n for _, c in shape) and not (shape & covered):
                    fill(covered | shape)

    fill(frozenset())
    return count


print([count_brute(n) for n in range(1, 9)])
assert [count_brute(n) for n in range(1, 9)] == [1, 2, 5, 11, 24, 53, 117, 258]
assert count_brute(3) == 5 and count_brute(1) == 1
# chắc chắn không sinh trùng: số cách lát liệt kê bằng số phần tử khác nhau
for n in range(1, 8):
    ts = all_tilings(n)
    assert len(ts) == len(set(ts))
print("mỗi cách lát được sinh đúng một lần (không trùng)")

[1, 2, 5, 11, 24, 53, 117, 258]
mỗi cách lát được sinh đúng một lần (không trùng)


**Độ phức tạp:** thời gian tỉ lệ với số cách lát `f(n) ≈ 2,2^n` — `n = 1000` hoàn toàn bất khả; dùng được tới `n ≈ 12` và đóng vai **oracle**.

**Nút thắt:** ta *liệt kê* từng cách lát trong khi chỉ cần *đếm*. Nhiều cách lát chia sẻ cùng phần nền bên trái; chỉ cần biết "phần nền bên trái đã lát xong và nhô ra ngoài những ô nào".

## 6. Ý tưởng thuật toán / bước đột phá

Ta lát từ **trái sang phải**. Sau khi quyết định xong các viên gạch có cột trái nhất nằm ở các cột `0..i−1`, các cột `0..i−1` đã đầy; cột `i` có thể đã bị phủ một số ô bởi các viên gạch nhô sang (tối đa tromino chiếm 2 cột). Vậy ta chỉ cần nhớ **cột `i` đã bị phủ những ô nào**:

| mask | ý nghĩa | tả |
|---|---|---|
| `0` | cột `i` còn trống hoàn toàn | |
| `1` | chỉ ô **trên** đã bị phủ | |
| `2` | chỉ ô **dưới** đã bị phủ | |
| `3` | cả hai ô đã bị phủ (**đầy**) | |

Gọi `dp[i][mask]` là số cách lát sao cho mọi cột `< i` đầy và cột `i` có mặt nạ phủ sẵn `mask`. Đáp án là `dp[n][0]` (cột `n` — ngoài bảng — không có ô nào bị phủ).

**Chuyển trạng thái** từ `dp[i][mask]` sang cột `i + 1`, bằng cách lấp các ô còn trống của cột `i` bằng các viên gạch có cột trái là `i`:

- `mask = 3` (cột `i` đầy): không đặt gì, sang `dp[i+1][0]`.
- `mask = 0` (trống): (a) domino đứng → `dp[i+1][0]`; (b) hai domino nằm → `dp[i+1][3]`; (c) tromino L phủ cả cột `i` và ô **trên** của cột `i+1` → `dp[i+1][1]`; (d) tromino L phủ cả cột `i` và ô **dưới** của cột `i+1` → `dp[i+1][2]`.
- `mask = 1` (trên đã phủ, dưới trống): ô dưới `(1,i)` phải được phủ: (a) domino nằm ở hàng dưới → cột `i+1` bị phủ ô dưới → `dp[i+1][2]`; (b) tromino L phủ `(1,i)`, `(1,i+1)`, `(0,i+1)` → cột `i+1` đầy → `dp[i+1][3]`.
- `mask = 2`: đối xứng: domino nằm hàng trên → `dp[i+1][1]`; tromino L → `dp[i+1][3]`.

Mọi trường hợp khác (ví dụ tromino chiếm một ô cột `i` và hai ô cột `i+1` khi cột `i` còn ô trống khác) đều **không** hoàn thành được vì ô trống còn lại của cột `i` không có viên gạch nào lấp mà không đè lên gạch kia. Bảng chuyển trạng thái này sẽ được kiểm chứng bằng oracle ở mục 12.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** DP theo trạng thái cột (bảng ở mục 6), **Cách 3** công thức truy hồi `f(n) = 2f(n−1) + f(n−3)`, **Cách 4** nhân ma trận.

### Cách 2 — DP theo trạng thái cột (cách "doocs")

**Giải thích từng đoạn:**

1. `dp[mask]` giữ số cách cho cột hiện tại, khởi tạo `dp = [1, 0, 0, 0]` (cột `0`, chưa phủ ô nào).
2. Lặp `n` lần: tính `new` từ `dp` theo bảng chuyển ở mục 6.
3. Đáp án `dp[0]` (cột `n` không có ô nào bị phủ), nhớ `% MOD` mỗi bước.

In [8]:
def num_tilings_states(n, mod=None):
    dp = [1, 0, 0, 0]                          # dp[mask] cho cột 0
    for _ in range(n):
        new = [0, 0, 0, 0]
        new[0] = dp[0] + dp[3]                 # từ trống: domino đứng; từ đầy: không đặt gì
        new[3] = dp[0] + dp[1] + dp[2]         # hai domino nằm (từ trống); L khép chuỗi (từ 1 hoặc 2)
        new[1] = dp[0] + dp[2]                 # L bậc 1 (từ trống); domino nằm hàng trên (từ 2)
        new[2] = dp[0] + dp[1]                 # L bậc 1 (từ trống); domino nằm hàng dưới (từ 1)
        dp = [x % mod for x in new] if mod else new
    return dp[0]


print([num_tilings_states(n) for n in range(0, 11)])
assert [num_tilings_states(n) for n in range(0, 9)] == [1, 1, 2, 5, 11, 24, 53, 117, 258]
assert num_tilings_states(3) == 5 and num_tilings_states(1) == 1

[1, 1, 2, 5, 11, 24, 53, 117, 258, 569, 1255]


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionStates:
    def numTilings(self, n: int) -> int:
        mod = 10 ** 9 + 7
        dp = [1, 0, 0, 0]                      # [rỗng, thừa ô trên, thừa ô dưới, đầy] của cột hiện tại
        for _ in range(n):
            e, t, b, f = dp
            dp = [(e + f) % mod, (e + b) % mod, (e + t) % mod, (e + t + b) % mod]
        return dp[0]

Ghi chú thứ tự: trong `SolutionStates` ta sắp `dp = [mask 0, mask 1, mask 2, mask 3]` và gọi `e, t, b, f` lần lượt là *trống, trên, dưới, đầy* nên `dp mới = [e + f, e + b, e + t, e + t + b]` đúng như các dòng `new[0..3]` ở trên.

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ (bốn số). 

### Cách 3 — Công thức truy hồi `f(n) = 2·f(n−1) + f(n−3)`

**Ý tưởng:** khử trạng thái để được một công thức duy nhất. Từ dãy ở mục 4 (`f = 1, 1, 2, 5, 11, 24, 53, …`) ta thấy `5 = 2·2 + 1`, `11 = 2·5 + 1`, `24 = 2·11 + 2`, `53 = 2·24 + 5`. Công thức: **`f(n) = 2·f(n−1) + f(n−3)`** với `f(0) = 1, f(1) = 1, f(2) = 2`. Chứng minh ở mục 10.

In [10]:
def num_tilings_recurrence(n, mod=None):
    f = [1, 1, 2]
    if n < 3:
        return f[n]
    a, b, c = 1, 1, 2                          # f(0), f(1), f(2)
    for _ in range(3, n + 1):
        a, b, c = b, c, (2 * c + a)            # f(i) = 2 f(i-1) + f(i-3)
        if mod:
            c %= mod
    return c


print([num_tilings_recurrence(n) for n in range(0, 11)])
assert [num_tilings_recurrence(n) for n in range(0, 9)] == [1, 1, 2, 5, 11, 24, 53, 117, 258]

[1, 1, 2, 5, 11, 24, 53, 117, 258, 569, 1255]


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionRecurrence:
    def numTilings(self, n: int) -> int:
        mod = 10 ** 9 + 7
        if n <= 2:
            return n                           # f(1) = 1, f(2) = 2
        f = [0] * (n + 1)
        f[0], f[1], f[2] = 1, 1, 2
        for i in range(3, n + 1):
            f[i] = (2 * f[i - 1] + f[i - 3]) % mod
        return f[n]

**Độ phức tạp:** `O(n)` thời gian; bộ nhớ `O(n)` (hoặc `O(1)` nếu cuộn ba biến như hàm `num_tilings_recurrence`).

### Cách 4 — Nhân ma trận (`O(log n)`)

Truy hồi bậc 3 `f(n) = 2f(n−1) + 0·f(n−2) + 1·f(n−3)` viết thành nhân ma trận:

```
[f(n)  ]   [2 0 1] [f(n-1)]
[f(n-1)] = [1 0 0] [f(n-2)]
[f(n-2)]   [0 1 0] [f(n-3)]
```

Vector đầu `(f(2), f(1), f(0)) = (2, 1, 1)`. Suy ra `(f(n), f(n−1), f(n−2))ᵀ = M^(n−2) · (2, 1, 1)ᵀ` với `n ≥ 2`. Luỹ thừa nhanh cho `O(log n)` — cần thiết nếu `n` lên tới `10^18`.

In [12]:
M = [[2, 0, 1], [1, 0, 0], [0, 1, 0]]


def num_tilings_matrix(n, mod=None):
    if n <= 2:
        return [1, 1, 2][n]
    P = mat_pow(M, n - 2, mod)
    v = [2, 1, 1]
    return sum(P[0][j] * v[j] for j in range(3)) % mod if mod else sum(P[0][j] * v[j] for j in range(3))


print([num_tilings_matrix(n) for n in range(0, 11)])
assert [num_tilings_matrix(n) for n in range(0, 9)] == [1, 1, 2, 5, 11, 24, 53, 117, 258]
print("f(10^18) mod 1e9+7 =", num_tilings_matrix(10 ** 18, MOD), "(tính trong vài micro giây)")

[1, 1, 2, 5, 11, 24, 53, 117, 258, 569, 1255]
f(10^18) mod 1e9+7 = 989096457 (tính trong vài micro giây)


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionMatrix:
    def numTilings(self, n: int) -> int:
        mod = 10 ** 9 + 7
        if n <= 2:
            return n

        def mul(A, B):
            return [[sum(A[i][k] * B[k][j] for k in range(3)) % mod for j in range(3)] for i in range(3)]

        result = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
        base = [[2, 0, 1], [1, 0, 0], [0, 1, 0]]
        e = n - 2
        while e:
            if e & 1:
                result = mul(result, base)
            base = mul(base, base)
            e >>= 1
        # (f(n), f(n-1), f(n-2)) = result * (f(2), f(1), f(0)) = result * (2, 1, 1)
        return (result[0][0] * 2 + result[0][1] * 1 + result[0][2] * 1) % mod

**Độ phức tạp:** `O(log n)` phép nhân ma trận `3 × 3` (hằng số 27); bộ nhớ `O(1)`. Với giới hạn `n ≤ 1000`, DP `O(n)` đã quá đủ, còn nhân ma trận để mở rộng (bài tập, biến thể `n` rất lớn).

## 8. TRACE từng bước

**Trace Cách 2** với `n = 3`. Mỗi hàng là một cột `i`; các số trong cột là `dp[i][mask]` (`mask = 0` trống, `1` thừa ô trên, `2` thừa ô dưới, `3` đầy):

| cột `i` | `dp[0]` | `dp[1]` | `dp[2]` | `dp[3]` | ghi chú |
|---|---|---|---|---|---|
| 0 | 1 | 0 | 0 | 0 | khởi tạo |
| 1 | 1 | 1 | 1 | 1 | từ `dp[0]=1`: đứng, L trên, L dưới, hai nằm |
| 2 | 2 | 2 | 2 | 3 | `dp[0]=1+1`, `dp[3]=1+1+1` |
| 3 | **5** | 5 | 5 | 7 | đáp án `dp[3][0] = dp[2][0] + dp[2][3] = 2 + 3 = 5` |

Hàm trace in từng bước kèm công thức từng ô (và đối chiếu bằng truy hồi):

In [14]:
def trace_states(n):
    dp = [1, 0, 0, 0]
    print(f"n = {n}: dp = [trống, thừa-trên, thừa-dưới, đầy]")
    print(f"cột 0: dp = {dp}")
    for i in range(1, n + 1):
        e, t, b, f = dp
        new = [e + f, e + b, e + t, e + t + b]
        print(f"cột {i}: dp[0] = {e}+{f} = {new[0]} | dp[1] = {e}+{b} = {new[1]} | dp[2] = {e}+{t} = {new[2]} | dp[3] = {e}+{t}+{b} = {new[3]}")
        dp = new
    print(f"=> đáp án dp[0] = {dp[0]}")
    return dp[0]


assert trace_states(3) == 5
print()
assert trace_states(5) == 24

n = 3: dp = [trống, thừa-trên, thừa-dưới, đầy]
cột 0: dp = [1, 0, 0, 0]
cột 1: dp[0] = 1+0 = 1 | dp[1] = 1+0 = 1 | dp[2] = 1+0 = 1 | dp[3] = 1+0+0 = 1
cột 2: dp[0] = 1+1 = 2 | dp[1] = 1+1 = 2 | dp[2] = 1+1 = 2 | dp[3] = 1+1+1 = 3
cột 3: dp[0] = 2+3 = 5 | dp[1] = 2+2 = 4 | dp[2] = 2+2 = 4 | dp[3] = 2+2+2 = 6
=> đáp án dp[0] = 5

n = 5: dp = [trống, thừa-trên, thừa-dưới, đầy]
cột 0: dp = [1, 0, 0, 0]
cột 1: dp[0] = 1+0 = 1 | dp[1] = 1+0 = 1 | dp[2] = 1+0 = 1 | dp[3] = 1+0+0 = 1
cột 2: dp[0] = 1+1 = 2 | dp[1] = 1+1 = 2 | dp[2] = 1+1 = 2 | dp[3] = 1+1+1 = 3
cột 3: dp[0] = 2+3 = 5 | dp[1] = 2+2 = 4 | dp[2] = 2+2 = 4 | dp[3] = 2+2+2 = 6
cột 4: dp[0] = 5+6 = 11 | dp[1] = 5+4 = 9 | dp[2] = 5+4 = 9 | dp[3] = 5+4+4 = 13
cột 5: dp[0] = 11+13 = 24 | dp[1] = 11+9 = 20 | dp[2] = 11+9 = 20 | dp[3] = 11+9+9 = 29
=> đáp án dp[0] = 24


**Trace Cách 3** (truy hồi) với `n = 6`: `f(0..2) = 1, 1, 2`; `f(3) = 2·2 + 1 = 5`; `f(4) = 2·5 + 1 = 11`; `f(5) = 2·11 + 2 = 24`; `f(6) = 2·24 + 5 = 53`. Hàm dưới in từng bước:

In [15]:
def trace_recurrence(n):
    f = [1, 1, 2]
    print("f(0), f(1), f(2) =", f)
    for i in range(3, n + 1):
        f.append(2 * f[i - 1] + f[i - 3])
        print(f"f({i}) = 2*f({i - 1}) + f({i - 3}) = 2*{f[i - 1]} + {f[i - 3]} = {f[i]}")
    return f[n]


assert trace_recurrence(6) == 53

f(0), f(1), f(2) = [1, 1, 2]
f(3) = 2*f(2) + f(0) = 2*2 + 1 = 5
f(4) = 2*f(3) + f(1) = 2*5 + 1 = 11
f(5) = 2*f(4) + f(2) = 2*11 + 2 = 24
f(6) = 2*f(5) + f(3) = 2*24 + 5 = 53


## 9. Code nộp LeetCode

Chọn **DP theo trạng thái cột** (cách "gốc" của bài, trực quan, `O(n)` thời gian, `O(1)` bộ nhớ). Giữ nguyên `class Solution`, `numTilings(self, n)`:

In [16]:
class Solution:
    def numTilings(self, n: int) -> int:
        mod = 10 ** 9 + 7
        # dp = [cột trống, thừa ô trên, thừa ô dưới, cột đầy] của cột hiện tại
        dp = [1, 0, 0, 0]
        for _ in range(n):
            e, t, b, f = dp
            dp = [
                (e + f) % mod,        # trống: domino đứng; đầy: không đặt gì
                (e + b) % mod,        # từ trống: tromino L (phủ ô trên cột kế); từ "thừa ô dưới": domino nằm hàng trên
                (e + t) % mod,        # đối xứng
                (e + t + b) % mod,    # từ trống: hai domino nằm; từ thừa 1 ô: tromino L khép lại
            ]
        return dp[0]


print(Solution().numTilings(3), Solution().numTilings(1))
assert Solution().numTilings(3) == 5 and Solution().numTilings(1) == 1

5 1


## 10. Vì sao đúng

**(a) Bảng chuyển trạng thái.** Tại cột `i` với mặt nạ `mask`, mọi viên gạch phủ một ô trống của cột `i` đều có cột trái nhất **bằng `i`** (vì các cột `< i` đã đầy). Ta xét **ô trống trên cùng** của cột `i`:

- `mask = 0`: cả hai ô của cột `i` còn trống. Ô trên `(0,i)` phải được phủ bởi: domino đứng (phủ luôn ô dưới); domino nằm hàng trên (khi đó ô dưới `(1,i)` chỉ có thể phủ bằng domino nằm hàng dưới, vì mọi viên khác phủ `(1,i)` đều đè lên `(0,i+1)`); hoặc tromino L phủ `(0,i)`: nếu L phủ cả cột `i` và một ô của cột `i+1` thì có đúng hai hướng (ô nhô ra ở hàng trên hoặc hàng dưới); nếu L phủ **một** ô của cột `i` và hai ô của cột `i+1` thì ô còn lại của cột `i` bị cô lập (không viên nào lấp được mà không đè lên L). Vậy chỉ còn **4** cách: đứng, hai nằm, L-trên, L-dưới.
- `mask = 1` hoặc `2`: chỉ còn một ô trống, phải phủ bằng domino nằm (cho ra `mask` mới là 2 hoặc 1 cho cột kế) hoặc L (đúng một hướng thích hợp) phủ ô đó và **cả hai** ô cột kế → `mask = 3`.
- `mask = 3`: không đặt gì.

Mỗi cách lát hoàn chỉnh tương ứng **đúng một** dãy chuyển trạng thái (vì ta lấp theo thứ tự cột và các lựa chọn là duy nhất tại mỗi bước), nên `dp` đếm đúng và không trùng. Bảng chuyển đã được kiểm chứng bằng oracle vét cạn.

**(b) Chứng minh `f(n) = 2f(n−1) + f(n−3)`.** Khử trạng thái: đặt `a_i = dp[i][0]`, `b_i = dp[i][3]`, `c_i = dp[i][1] = dp[i][2]`. Từ bảng chuyển: `a_{i+1} = a_i + b_i`, `b_{i+1} = a_i + 2c_i`, `c_{i+1} = a_i + c_i`. Dùng cách đếm trực tiếp theo cột cuối: `f(n) = f(n−1) + f(n−2) + 2·Σ_{k=0}^{n−3} f(k)` (cột trái nhất: domino đứng, hai domino nằm, hoặc một chuỗi bắt đầu bằng L (hai hướng) dài `≥ 3` cột rồi khép lại, phần còn lại là `f(k)`). Viết thêm cho `n−1` rồi **trừ**:

```
f(n)   = f(n-1) + f(n-2) + 2·S(n-3)
f(n-1) = f(n-2) + f(n-3) + 2·S(n-4)
-------------------------------------- trừ vế
f(n) − f(n−1) = f(n−1) − f(n−2) + f(n−2) − f(n−3) + 2f(n−3)
              = f(n−1) + f(n−3)        ⇒   f(n) = 2f(n−1) + f(n−3).   ∎
```

Ta đồng thời kiểm chứng công thức tổng `f(n) = f(n−1) + f(n−2) + 2Σ f(k)` bằng máy ở mục 12.

**(c) Modulo.** Mọi bước chỉ dùng `+` và `×` nên lấy modulo sau mỗi bước cho đúng kết quả mod; ta so khớp với số nguyên lớn chính xác `% MOD` ở mục 12.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê quay lui | `O(f(n)) ≈ O(2,2^n)` | `O(n)` | chỉ làm oracle (`n ≤ 12`) |
| 2. DP trạng thái cột | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Truy hồi `2f(n−1)+f(n−3)` | `O(n)` | `O(1)`/`O(n)` | ngắn gọn nhất |
| 4. Nhân ma trận | `O(log n)` | `O(1)` | khi `n` rất lớn |

Thời gian thực tế (đo bằng `perf_counter`):

In [17]:
def bench(fn, n, repeat=5):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(n)
        best = min(best, time.perf_counter() - t0)
    return best


for n in (10, 12):
    print(f"n = {n}: brute {bench(count_brute, n, 1) * 1000:9.2f} ms")
for n in (1000, 10 ** 5):
    row = [f"states {bench(lambda x: num_tilings_states(x, MOD), n) * 1000:8.3f} ms",
           f"recurrence {bench(lambda x: num_tilings_recurrence(x, MOD), n) * 1000:8.3f} ms",
           f"matrix {bench(lambda x: num_tilings_matrix(x, MOD), n) * 1000:8.4f} ms"]
    print(f"n = {n:6d}: " + " | ".join(row))

n = 10: brute     65.23 ms


n = 12: brute    340.93 ms
n =   1000: states    0.686 ms | recurrence    0.141 ms | matrix   0.1725 ms


n = 100000: states   39.327 ms | recurrence    8.627 ms | matrix   0.1334 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **vét cạn** liệt kê cách lát và đối chiếu `n = 1..12`, (E) so sánh các cách trên toàn bộ `n = 1..1000` kèm số nguyên chính xác, (F) giới hạn lớn nhất và `n` cực lớn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "states": lambda n: num_tilings_states(n, MOD),
    "recurrence": lambda n: num_tilings_recurrence(n, MOD),
    "matrix": lambda n: num_tilings_matrix(n, MOD),
    "Solution": lambda n: Solution().numTilings(n),
    "SolutionStates": lambda n: SolutionStates().numTilings(n),
    "SolutionRecurrence": lambda n: SolutionRecurrence().numTilings(n),
    "SolutionMatrix": lambda n: SolutionMatrix().numTilings(n),
}


def check(n, want, names=None):
    for name in (names or IMPLS):
        got = IMPLS[name](n)
        assert got == want, (name, n, got, want)


check(3, 5)
check(1, 1)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 7 cài đặt đúng


### B. Edge case

- `n = 1` → `1`; `n = 2` → `2` (hai domino đứng, hoặc hai domino nằm).
- Các giá trị đầu dãy `n = 1..8`: `1, 2, 5, 11, 24, 53, 117, 258`.
- `n` ngay sau khi đáp án bắt đầu vượt `10^9 + 7` (khoảng `n = 26`): kiểm tra phép modulo bắt đầu có tác dụng.
- `n = 1000`: giới hạn lớn nhất.

In [19]:
head = {1: 1, 2: 2, 3: 5, 4: 11, 5: 24, 6: 53, 7: 117, 8: 258, 9: 569, 10: 1255}
for n, want in head.items():
    check(n, want)

exact = [1, 1, 2]
for i in range(3, 1001):
    exact.append(2 * exact[i - 1] + exact[i - 3])
first_over = next(n for n in range(1, 1001) if exact[n] >= MOD)
print("f(n) lần đầu >= 10^9+7 tại n =", first_over, "| f(n) =", exact[first_over], "-> mod =", exact[first_over] % MOD)
for n in (first_over - 1, first_over, first_over + 1):
    check(n, exact[n] % MOD)
check(1000, exact[1000] % MOD)
print("f(1000) có", len(str(exact[1000])), "chữ số; f(1000) mod 1e9+7 =", exact[1000] % MOD)
print("edge case OK")

f(n) lần đầu >= 10^9+7 tại n = 28 | f(n) = 1914332891 -> mod = 914332884
f(1000) có 344 chữ số; f(1000) mod 1e9+7 = 979232805
edge case OK


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_domino_only(n):                          # chỉ domino: Fibonacci
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b % MOD


def bug_missing_factor_two(n):                   # quên hai hướng của tromino: f(n)=f(n-1)+f(n-2)+sum f(k)
    f = [1, 1]
    for i in range(2, n + 1):
        f.append(f[i - 1] + f[i - 2] + (sum(f[:i - 2]) if i >= 3 else 0))
    return f[n] % MOD


def bug_base_zero(n):                            # f(0) = 0 thay vì 1
    f = [0, 1, 2]
    for i in range(3, n + 1):
        f.append((2 * f[i - 1] + f[i - 3]) % MOD)
    return f[n]


def bug_merge_states(n):                         # gộp "thừa ô trên" và "thừa ô dưới" thành một trạng thái, đếm một lần
    e, h, f = 1, 0, 0
    for _ in range(n):
        e, h, f = e + f, e + h, e + h
    return e % MOD


def bug_no_mod_small(n):                         # mod ở dòng đầu nhưng quên ở nhánh cộng: tràn khi so sánh
    f = [1, 1, 2]
    for i in range(3, n + 1):
        f.append(2 * f[i - 1] + f[i - 3])
    return f[n]                                  # QUÊN % MOD


BUGS = [bug_domino_only, bug_missing_factor_two, bug_base_zero, bug_merge_states, bug_no_mod_small]
for bug in BUGS:
    bad = next(n for n in range(1, 60) if bug(n) != exact[n] % MOD)
    print(f"{bug.__name__:24s} sai đầu tiên tại n = {bad}: {bug(bad)} (đúng {exact[bad] % MOD})")

bug_domino_only          sai đầu tiên tại n = 3: 3 (đúng 5)
bug_missing_factor_two   sai đầu tiên tại n = 3: 4 (đúng 5)
bug_base_zero            sai đầu tiên tại n = 3: 4 (đúng 5)
bug_merge_states         sai đầu tiên tại n = 3: 4 (đúng 5)
bug_no_mod_small         sai đầu tiên tại n = 28: 1914332891 (đúng 914332884)


### D. Vét cạn: liệt kê cách lát thật và đối chiếu

Dùng `count_brute` (đếm bằng quay lui, **độc lập** với mọi công thức) cho `n = 1..12` và so với tất cả cài đặt. Đồng thời kiểm chứng công thức tổng ở mục 10: `f(n) = f(n−1) + f(n−2) + 2·Σ_{k=0}^{n−3} f(k)`.

In [21]:
brute = {n: count_brute(n) for n in range(0, 13)}
print("brute:", [brute[n] for n in range(0, 13)])
for n in range(1, 13):
    check(n, brute[n] % MOD)
for n in range(3, 13):
    assert brute[n] == brute[n - 1] + brute[n - 2] + 2 * sum(brute[k] for k in range(0, n - 2)), n
    assert brute[n] == 2 * brute[n - 1] + brute[n - 3], n
print("n = 1..12: mọi cài đặt khớp vét cạn; công thức tổng và truy hồi đều đúng trên số liệu vét cạn")

# kiểm chứng thêm: chỉ domino cho Fibonacci; chỉ tromino L thì n phải chia hết cho 3
def count_with(shape_filter, n):
    count = 0

    def fill(covered):
        nonlocal count
        first = next(((r, c) for c in range(n) for r in (0, 1) if (r, c) not in covered), None)
        if first is None:
            count += 1
            return
        c0 = first[1]
        for cc in ((c0, c0 - 1) if c0 > 0 else (c0,)):
            for shape in shapes_at(cc):
                if shape_filter(shape) and first in shape and all(0 <= c < n for _, c in shape) and not (shape & covered):
                    fill(covered | shape)

    fill(frozenset())
    return count


dominoes_only = [count_with(lambda s: len(s) == 2, n) for n in range(0, 11)]
print("chỉ domino:", dominoes_only, "(Fibonacci)")
assert dominoes_only == [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]
trominoes_only = [count_with(lambda s: len(s) == 3, n) for n in range(0, 10)]
print("chỉ tromino L:", trominoes_only, "(chỉ khác 0 khi n chia hết cho 3: 2^(n/3) cách)")

brute: [1, 1, 2, 5, 11, 24, 53, 117, 258, 569, 1255, 2768, 6105]
n = 1..12: mọi cài đặt khớp vét cạn; công thức tổng và truy hồi đều đúng trên số liệu vét cạn
chỉ domino: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89] (Fibonacci)
chỉ tromino L: [1, 0, 0, 2, 0, 0, 4, 0, 0, 8] (chỉ khác 0 khi n chia hết cho 3: 2^(n/3) cách)


### E. So sánh các cách trên mọi `n = 1..1000`

Số nguyên chính xác `exact[n]` (không modulo, do Python) được tính bằng truy hồi; ta so `exact[n] % MOD` với **mọi** cài đặt cho **mọi** `n` từ `1` đến `1000`, và cũng kiểm chứng truy hồi bằng DP trạng thái **chính xác** (không modulo).

In [22]:
# DP trạng thái chính xác (không modulo) so với truy hồi chính xác
exact_states = [num_tilings_states(n) for n in range(0, 1001)]
assert exact_states == exact
print("DP trạng thái (số nguyên chính xác) = truy hồi chính xác trên n = 0..1000")

for n in range(1, 1001):
    want = exact[n] % MOD
    for name, fn in IMPLS.items():
        assert fn(n) == want, (name, n)
print("n = 1..1000: 7 cài đặt khớp số chính xác mod 10^9+7 trên cả 1000 giá trị")

# tính chất: truy hồi bậc 3 cũng đúng sau modulo
vals = [num_tilings_states(n, MOD) for n in range(0, 1001)]
assert all(vals[n] == (2 * vals[n - 1] + vals[n - 3]) % MOD for n in range(3, 1001))
print("f(n) = 2 f(n-1) + f(n-3) (mod) trên toàn dãy 0..1000")

DP trạng thái (số nguyên chính xác) = truy hồi chính xác trên n = 0..1000


n = 1..1000: 7 cài đặt khớp số chính xác mod 10^9+7 trên cả 1000 giá trị


f(n) = 2 f(n-1) + f(n-3) (mod) trên toàn dãy 0..1000


### F. Giới hạn lớn nhất và `n` cực lớn

`n = 1000` đã kiểm ở E. Với `n` cực lớn (mở rộng, ngoài ràng buộc của đề) chỉ cách nhân ma trận dùng được. Ta kiểm chứng **tính nhất quán** của nhân ma trận: `f(a + b)` tính bằng nhân hai ma trận bằng với tính trực tiếp.

In [23]:
rng = random.Random(790)
for _ in range(200):
    n = rng.randint(3, 1000)
    assert num_tilings_matrix(n, MOD) == exact[n] % MOD
    a, b = rng.randint(2, 500), rng.randint(2, 500)
    Pa, Pb, Pab = mat_pow(M, a, MOD), mat_pow(M, b, MOD), mat_pow(M, a + b, MOD)
    assert mat_mul(Pa, Pb, MOD) == Pab
print("nhân ma trận nhất quán: M^a · M^b = M^(a+b) (mod) và khớp truy hồi trên 200 n ngẫu nhiên")
for n in (10 ** 6, 10 ** 12, 10 ** 18):
    t0 = time.perf_counter()
    r = num_tilings_matrix(n, MOD)
    print(f"f(10^{len(str(n)) - 1}) mod 1e9+7 = {r}   ({(time.perf_counter() - t0) * 1e6:.0f} µs)")
assert num_tilings_matrix(10 ** 6, MOD) == num_tilings_states(10 ** 6, MOD)

nhân ma trận nhất quán: M^a · M^b = M^(a+b) (mod) và khớp truy hồi trên 200 n ngẫu nhiên
f(10^6) mod 1e9+7 = 868423319   (178 µs)
f(10^12) mod 1e9+7 = 83703756   (367 µs)
f(10^18) mod 1e9+7 = 989096457   (584 µs)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ dùng domino (ra dãy Fibonacci)

Quên tromino. `f(n) = f(n−1) + f(n−2)` cho `1, 2, 3, 5, 8, …`: sai ngay từ `n = 3` (đúng là `5`).

In [24]:
print("domino only n=1..8:", [bug_domino_only(n) for n in range(1, 9)])
print("đúng            :", [Solution().numTilings(n) for n in range(1, 9)])
assert bug_domino_only(3) == 3 and Solution().numTilings(3) == 5

domino only n=1..8: [1, 2, 3, 5, 8, 13, 21, 34]
đúng            : [1, 2, 5, 11, 24, 53, 117, 258]


**Vì sao sai:** tromino L thêm vào các "chuỗi" cột liên kết nhau; hai tromino L đối xứng đặt đầu cột và domino nằm ở giữa là cách lát hợp lệ mà domino thuần không tạo ra. **Đúng:** thêm trạng thái "thừa ô" (hoặc dùng `2f(n−1) + f(n−3)`).

### Lỗi 2: quên nhân đôi vì tromino có hai hướng

Mỗi chuỗi bắt đầu bằng tromino L có **hai** đối xứng (lật lên/xuống). Quên hệ số `2` trước tổng làm đếm thiếu.

In [25]:
print("quên hệ số 2, n=1..8:", [bug_missing_factor_two(n) for n in range(1, 9)])
assert bug_missing_factor_two(3) == 4 != 5

quên hệ số 2, n=1..8: [1, 2, 4, 8, 16, 32, 64, 128]


**Vì sao sai:** `n = 3` có 2 cách chứa L (L lật lên và L lật xuống), nhưng ta chỉ đếm 1 → ra `4`. **Đúng:** `f(n) = f(n−1) + f(n−2) + 2·Σ f(k)`.

### Lỗi 3: điều kiện đầu `f(0) = 0`

Bảng rỗng có **đúng 1** cách lát (không đặt viên nào). Đặt `f(0) = 0` làm `f(3) = 2·2 + 0 = 4` sai.

In [26]:
print("f(0)=0:", [bug_base_zero(n) for n in range(1, 8)], "| đúng:", [exact[n] for n in range(1, 8)])
assert bug_base_zero(3) == 4

f(0)=0: [1, 2, 4, 9, 20, 44, 97] | đúng: [1, 2, 5, 11, 24, 53, 117]


**Vì sao sai:** `f(n−3)` với `n = 3` là `f(0)`, tức "lát phần còn lại rỗng", phải bằng `1`. **Đúng:** `f = [1, 1, 2]`.

### Lỗi 4: gộp hai trạng thái "thừa ô trên" và "thừa ô dưới"

Hai trạng thái này đối xứng nên số cách bằng nhau, nhưng chúng là **hai** trạng thái riêng: từ trạng thái trống có **hai** cách đặt L (lên/xuống). Nếu gộp thành một và chỉ đếm một lần ta thiếu một nửa.

In [27]:
print("gộp trạng thái:", [bug_merge_states(n) for n in range(1, 8)], "| đúng:", [exact[n] for n in range(1, 8)])
assert bug_merge_states(3) != 5

gộp trạng thái: [1, 2, 4, 8, 16, 32, 64] | đúng: [1, 2, 5, 11, 24, 53, 117]


**Vì sao sai:** phép chuyển `e → (thừa-trên) + (thừa-dưới)` cần cộng **cả hai**; nếu chỉ có một trạng thái chung thì phải nhân đôi hệ số khi chuyển vào/ra nó. **Đúng:** giữ hai trạng thái, hoặc dùng `2×`.

### Lỗi 5: quên modulo

Không có `% MOD` thì các ngôn ngữ có số nguyên cố định tràn; Python không tràn nhưng kết quả **không khớp** đáp án mong đợi (và số lớn dần làm chậm).

In [28]:
print("n=40 không mod:", bug_no_mod_small(40), "| đúng (mod):", Solution().numTilings(40))
print("số chữ số của f(1000) nếu quên mod:", len(str(bug_no_mod_small(1000))))
assert bug_no_mod_small(40) != Solution().numTilings(40)

n=40 không mod: 25366833951139 | đúng (mod): 833773577
số chữ số của f(1000) nếu quên mod: 344


**Vì sao sai:** đề yêu cầu kết quả `% (10^9 + 7)`. **Đúng:** lấy modulo **ở mỗi bước** (hoặc cuối cùng nếu dùng số nguyên lớn của Python).

### Lỗi 6 (bonus): đệ quy không nhớ

Viết `f(n) = 2f(n−1) + f(n−3)` bằng đệ quy thuần không nhớ cho thời gian mũ `~2,2^n`; với `n = 1000` không bao giờ xong. Dùng vòng lặp hoặc `lru_cache`.

In [29]:
def f_rec(n):
    return [1, 1, 2][n] if n <= 2 else 2 * f_rec(n - 1) + f_rec(n - 3)


t0 = time.perf_counter()
print("f_rec(22) =", f_rec(22), f"({(time.perf_counter() - t0) * 1000:.0f} ms) — mỗi n tăng 1 thời gian nhân ~2,2")


@lru_cache(maxsize=None)
def f_memo(n):
    return [1, 1, 2][n] if n <= 2 else 2 * f_memo(n - 1) + f_memo(n - 3)


print("f_memo(300) mod:", f_memo(300) % MOD, "| đúng:", exact[300] % MOD)
assert f_memo(300) % MOD == exact[300] % MOD

f_rec(22) = 16630031 (0 ms) — mỗi n tăng 1 thời gian nhân ~2,2
f_memo(300) mod: 768506587 | đúng: 768506587


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm số cách lát/ghép/xếp trên dải rộng cố định (`2 × n`, `3 × n`)" → **DP theo cột với trạng thái là phần nhô ra** (profile DP). Chiều rộng càng nhỏ, số trạng thái càng ít (`2^{chiều cao}`).

**Mẹo:**

1. Lát theo thứ tự cố định (trái sang phải, trên xuống dưới) để mỗi cách lát chỉ sinh **một lần**.
2. Trạng thái = **cột hiện tại đã bị phủ ô nào**; hỏi "viên nào lớn nhất vươn sang cột kế?" để biết cần bao nhiêu cột.
3. Viết **brute force liệt kê** cho `n` nhỏ trước; từ dãy số suy đoán và kiểm chứng công thức (`1, 2, 5, 11, 24, 53, …`).
4. Từ DP trạng thái có thể **khử trạng thái** ra truy hồi tuyến tính, rồi **nhân ma trận** cho `n` cực lớn.
5. Luôn nhớ `f(0) = 1` (một cách lát bảng rỗng) và kiểm tra modulo ở mọi bước.
6. Sinh dãy đầu (OEIS-style) bằng brute rồi dùng `assert` để chứng minh DP.

**Họ bài cùng pattern:** Climbing Stairs (70), Domino Tiling `3 × n`, Number of Ways to Paint `N × 3` Grid (1411), Count Vowels Permutation (1220), Knight Dialer (935) — các bài "chuyển trạng thái tuyến tính" đều làm được bằng nhân ma trận.

## 15. Biến thể

1. **Chỉ domino** `2 × n`: Fibonacci (`f(n) = f(n−1) + f(n−2)`).
2. **Domino `3 × n`**: `0` khi `n` lẻ; chuỗi `1, 3, 11, 41, …` (profile DP với `2^3 = 8` trạng thái).
3. **`n` rất lớn** (`10^18`): nhân ma trận.
4. **Thêm ô vuông `1 × 1`** (mỗi ô trống có thể để vuông): cần thêm hệ số.
5. **Đếm số cách khác nhau theo *đối xứng*** (loại cách lát trùng sau lật/quay): cần Burnside — nằm ngoài đề.

Biến thể 2 (domino `3 × n`) bằng profile DP tổng quát cho bảng `h × n`, đối chiếu với giá trị đã biết:

In [30]:
def domino_tilings(h, n):
    """Số cách lát bảng h x n bằng domino 2x1 (profile DP: mask = các ô của cột hiện tại đã bị phủ)."""
    full = (1 << h) - 1

    @lru_cache(maxsize=None)
    def go(col, mask):
        if col == n:
            return 1 if mask == 0 else 0
        # lấp cột col: đặt các domino đứng trong cột hoặc nằm sang cột kế; tạo next_mask
        res = 0

        def place(row, cur, nxt):
            nonlocal res
            if row == h:
                res += go(col + 1, nxt)
                return
            if cur >> row & 1:
                place(row + 1, cur, nxt)
                return
            # domino nằm: ô (row, col) và (row, col+1)
            if col + 1 < n:
                place(row + 1, cur | 1 << row, nxt | 1 << row)
            # domino đứng: ô (row, col) và (row+1, col)
            if row + 1 < h and not (cur >> (row + 1) & 1):
                place(row + 2, cur | 1 << row | 1 << (row + 1), nxt)

        place(0, mask, 0)
        return res

    return go(0, 0)


print("2 x n:", [domino_tilings(2, n) for n in range(0, 10)], "(Fibonacci)")
print("3 x n:", [domino_tilings(3, n) for n in range(0, 11)], "(0 khi n lẻ; 1, 3, 11, 41, 153)")
assert [domino_tilings(3, n) for n in range(0, 9)] == [1, 0, 3, 0, 11, 0, 41, 0, 153]
assert [domino_tilings(2, n) for n in range(0, 8)] == [1, 1, 2, 3, 5, 8, 13, 21]
print("8 x 8 (bàn cờ vua):", domino_tilings(8, 8), "(đáp án nổi tiếng 12988816)")
assert domino_tilings(8, 8) == 12988816

2 x n: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55] (Fibonacci)
3 x n: [1, 0, 3, 0, 11, 0, 41, 0, 153, 0, 571] (0 khi n lẻ; 1, 3, 11, 41, 153)
8 x 8 (bàn cờ vua): 12988816 (đáp án nổi tiếng 12988816)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê quay lui | Lấp ô trống đầu tiên bằng mọi viên khả dĩ | `O(f(n)) ≈ 2,2^n` | `O(n)` | Oracle, `n ≤ 12`; sinh dãy đầu để đoán quy luật |
| 2. DP trạng thái cột | `dp[mask]` của cột hiện tại | `O(n)` | `O(1)` | **Đáp án chính**; dễ mở rộng sang các bài lát khác |
| 3. Truy hồi `2f(n−1)+f(n−3)` | Khử trạng thái | `O(n)` | `O(1)` | Ngắn nhất; cần chứng minh trước khi tin |
| 4. Nhân ma trận | Truy hồi → `M^n` | `O(log n)` | `O(1)` | `n` rất lớn (`10^18`) |

Cả bốn khớp nhau trên `n = 1..1000` (số chính xác mod `10^9+7`) và `n = 1..12` vét cạn (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [31]:
f = [1, 1, 2]
for i in range(3, 8):
    f.append(2 * f[i - 1] + f[i - 3])
print(f)
print(f[7] % 100, sum(f))

[1, 1, 2, 5, 11, 24, 53, 117]
17 214


### Bài 2 (tính tay)

- (a) Dùng truy hồi `f(n) = 2f(n−1) + f(n−3)` tính `f(4)`, `f(5)`, `f(6)` từ `f(0..3) = 1, 1, 2, 5`.
- (b) Kiểm tra công thức tổng: `f(5) = f(4) + f(3) + 2(f(0) + f(1) + f(2))`.
- (c) Trong `11` cách lát `2 × 4`, bao nhiêu cách có chứa tromino L? Vẽ tay một cách (gợi ý: hai tromino L ghép thành khối `2 × 3`, cộng một domino đứng).

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP trạng thái cột:

In [32]:
def tilings_todo(n):
    mod = 10 ** 9 + 7
    dp = [1, 0, 0, 0]                  # [trống, thừa ô trên, thừa ô dưới, đầy]
    for _ in range(n):
        e, t, b, f = dp
        # TODO: tính dp mới theo bảng chuyển ở mục 6 (nhớ % mod)
        pass
    return None


# Khi làm xong, bỏ comment để kiểm tra:
# assert [tilings_todo(n) for n in range(1, 9)] == [1, 2, 5, 11, 24, 53, 117, 258]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` dùng công thức `f(n) = 2f(n−1) + f(n−3)`:

In [33]:
class MySolution:
    def numTilings(self, n):
        pass            # TODO


# Khi làm xong:
# for n, w in [(1, 1), (2, 2), (3, 5), (4, 11), (30, 312342182)]:
#     assert MySolution().numTilings(n) == w

### Bài 5 (mini challenge): `n` cực lớn

Viết `tilings_big(n)` trả về `numTilings(n)` với `n` tới `10^18` (đề gốc chỉ cho `n ≤ 1000` nên cần cách khác DP tuyến tính): dùng **nhân ma trận `3 × 3` và luỹ thừa nhanh** (hoặc luỹ thừa nhanh trên vector truy hồi).

In [34]:
def tilings_big(n):
    pass            # TODO


# Khi làm xong:
# assert tilings_big(1000) == Solution().numTilings(1000)
# assert tilings_big(10 ** 18) == 989096457

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: áp dụng `f(i) = 2f(i−1) + f(i−3)` từng bước.
- Mức 2: `f(3) = 5`, `f(4) = 11`.
- Mức 3: `f = [1, 1, 2, 5, 11, 24, 53, 117]`; `f(7) % 100 = 17`; tổng `= 214`.

**Bài 2**

- Mức 1: `f(4) = 2·5 + f(1) = 11`.
- Mức 2: `f(5) = 2·11 + f(2) = 24`; `f(6) = 2·24 + f(3) = 53`.
- Mức 3: (b) `11 + 5 + 2·(1 + 1 + 2) = 24` ✓. (c) ví dụ: hai tromino L ghép thành khối `2 × 3` (có 2 cách ghép), cộng một domino đứng ở cột còn lại (đặt trái hoặc phải); máy sẽ đếm chính xác ở mục 19.

**Bài 3**

- Mức 1: bốn số mới từ `e, t, b, f`.
- Mức 2: mới `[0] = e + f`, `[1] = e + b`, `[2] = e + t`, `[3] = e + t + b`.
- Mức 3: `dp = [(e + f) % mod, (e + b) % mod, (e + t) % mod, (e + t + b) % mod]`, rồi `return dp[0]`.

**Bài 4**

- Mức 1: `n <= 2` thì trả `n`.
- Mức 2: mảng `f` với `f[0..2] = 1, 1, 2`.
- Mức 3: `f[i] = (2 * f[i-1] + f[i-3]) % mod`.

**Bài 5**

- Mức 1: viết truy hồi dưới dạng nhân ma trận (mục 7, Cách 4).
- Mức 2: luỹ thừa nhanh `M^(n−2)`.
- Mức 3: kết quả `= (P[0][0]·2 + P[0][1] + P[0][2]) % mod` với `P = M^(n−2)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [35]:
f = [1, 1, 2]
for i in range(3, 8):
    f.append(2 * f[i - 1] + f[i - 3])
assert f == [1, 1, 2, 5, 11, 24, 53, 117]
assert f[7] % 100 == 17 and sum(f) == 214
print("Bài 1 OK:", f)

Bài 1 OK: [1, 1, 2, 5, 11, 24, 53, 117]


### Bài 2

(a) `f(4) = 11`, `f(5) = 24`, `f(6) = 53`. (b) `11 + 5 + 2·4 = 24` ✓. (c) Máy liệt kê `6` trong `11` cách có chứa tromino L (xem bên dưới); ví dụ hai tromino L ghép thành khối `2 × 3` ở bên trái, cộng một domino đứng ở cột cuối.

In [36]:
f = [1, 1, 2, 5]
for i in range(4, 7):
    f.append(2 * f[i - 1] + f[i - 3])
assert f[4:] == [11, 24, 53]
assert f[5] == f[4] + f[3] + 2 * (f[0] + f[1] + f[2])
ts = all_tilings(4)
with_L = [t for t in ts if any(len(tile) == 3 for tile in t)]
print("n=4: tổng", len(ts), "cách; có chứa tromino L:", len(with_L), "cách")
for t in with_L[:3]:
    print(" / ".join(draw(t, 4)))
assert len(ts) == 11 and len(with_L) == 6
print("Bài 2 OK")

n=4: tổng 11 cách; có chứa tromino L: 6 cách
ABCC / ABBC
ABBC / ABCC
ABBC / AACC
Bài 2 OK


### Bài 3

In [37]:
def tilings_answer(n):
    mod = 10 ** 9 + 7
    dp = [1, 0, 0, 0]
    for _ in range(n):
        e, t, b, f = dp
        dp = [(e + f) % mod, (e + b) % mod, (e + t) % mod, (e + t + b) % mod]
    return dp[0]


assert [tilings_answer(n) for n in range(1, 9)] == [1, 2, 5, 11, 24, 53, 117, 258]
assert all(tilings_answer(n) == exact[n] % MOD for n in range(1, 1001))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def numTilings(self, n):
        mod = 10 ** 9 + 7
        if n <= 2:
            return n
        f = [1, 1, 2]
        for i in range(3, n + 1):
            f.append((2 * f[i - 1] + f[i - 3]) % mod)
        return f[n]


for n, w in [(1, 1), (2, 2), (3, 5), (4, 11), (30, exact[30] % MOD)]:
    assert MySolutionAnswer().numTilings(n) == w
print("f(30) mod 1e9+7 =", exact[30] % MOD)
assert all(MySolutionAnswer().numTilings(n) == exact[n] % MOD for n in range(1, 1001))
print("Bài 4 OK")

f(30) mod 1e9+7 = 312342182


Bài 4 OK


### Bài 5

In [39]:
def tilings_big_answer(n):
    mod = 10 ** 9 + 7
    if n <= 2:
        return n
    P = mat_pow([[2, 0, 1], [1, 0, 0], [0, 1, 0]], n - 2, mod)
    return (P[0][0] * 2 + P[0][1] + P[0][2]) % mod


assert tilings_big_answer(1000) == Solution().numTilings(1000)
assert all(tilings_big_answer(n) == exact[n] % MOD for n in range(1, 1001))
val = tilings_big_answer(10 ** 18)
print("f(10^18) mod 1e9+7 =", val)
assert val == num_tilings_matrix(10 ** 18, MOD)
print("Bài 5 OK")

f(10^18) mod 1e9+7 = 989096457
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (lát `2 × n`, DP theo trạng thái cột):**

```python
class Solution:
    def numTilings(self, n: int) -> int:
        mod = 10 ** 9 + 7
        dp = [1, 0, 0, 0]                  # [trống, thừa ô trên, thừa ô dưới, đầy]
        for _ in range(n):
            e, t, b, f = dp
            dp = [(e + f) % mod, (e + b) % mod, (e + t) % mod, (e + t + b) % mod]
        return dp[0]
```

**Công thức nhanh:** `f(0), f(1), f(2) = 1, 1, 2`; `f(n) = 2·f(n−1) + f(n−3)`.

**Checklist trước khi submit:**

- [ ] Đã tính **cả** domino và tromino L (hai hướng lật lên/xuống)?
- [ ] Điều kiện đầu `f(0) = 1` (bảng rỗng), `f(1) = 1`, `f(2) = 2`?
- [ ] Hai trạng thái "thừa ô trên", "thừa ô dưới" được giữ **riêng** (hoặc có hệ số 2)?
- [ ] Lấy `% (10^9 + 7)` ở mỗi bước?
- [ ] Đã đối chiếu dãy `1, 2, 5, 11, 24, 53, 117, 258` với liệt kê vét cạn?
- [ ] Đúng tên `class Solution`, `numTilings(self, n)`, không in debug.